# GPU Baseline Training

This notebook trains a solid GPU-ready baseline for speech emotion recognition using the preprocessed manifest.

What it does:
1. Loads `manifests/processed/processed_audio_manifest.csv`
2. Builds train/val/test datasets from the existing speaker-disjoint split
3. Computes log-spectrogram features inside the model
4. Trains a compact 2D CNN on GPU when CUDA is available
5. Saves the best checkpoint and reports validation/test macro-F1

If PyTorch is not installed yet, install a CUDA build first. Example:

```bash
pip install torch --index-url https://download.pytorch.org/whl/cu121
```

In [ ]:
from __future__ import annotations

import json
import math
import random
import wave
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset
except Exception as exc:
    raise ImportError(
        "PyTorch is required for this notebook. Install a CUDA-enabled build first, "
        "for example: pip install torch --index-url https://download.pytorch.org/whl/cu121"
    ) from exc


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "AudioWAV").exists() and (candidate / "context.md").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing AudioWAV/ and context.md")


ROOT = find_repo_root(Path.cwd().resolve())
MANIFEST_PATH = ROOT / "manifests" / "processed" / "processed_audio_manifest.csv"
ARTIFACT_DIR = ROOT / "artifacts" / "baseline_gpu"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
CLIP_SECONDS = 5.0
SAMPLE_RATE = 16_000
MAX_SAMPLES = int(CLIP_SECONDS * SAMPLE_RATE)
BATCH_SIZE = 64
NUM_WORKERS = 0  # Windows/Jupyter-safe default
EPOCHS = 20
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
DROPOUT = 0.25
USE_AMP = torch.cuda.is_available()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Repo root: {ROOT}")
print(f"Manifest: {MANIFEST_PATH}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
manifest["processed_file_path"] = manifest["processed_file_path"].map(lambda p: str(Path(p)))
manifest = manifest.loc[manifest["status"] == "ok"].copy()

label_names = sorted(manifest["emotion"].unique())
label_to_index = {label: idx for idx, label in enumerate(label_names)}
manifest["label"] = manifest["emotion"].map(label_to_index)

train_df = manifest.loc[manifest["split"] == "train"].reset_index(drop=True)
val_df = manifest.loc[manifest["split"] == "val"].reset_index(drop=True)
test_df = manifest.loc[manifest["split"] == "test"].reset_index(drop=True)

print("Label map:", label_to_index)
print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))
train_df[["emotion", "split"]].value_counts().sort_index()

In [ ]:
def read_pcm16_mono_wav(path: str) -> np.ndarray:
    with wave.open(str(path), "rb") as wav_file:
        channels = wav_file.getnchannels()
        sample_rate = wav_file.getframerate()
        sample_width = wav_file.getsampwidth()
        n_frames = wav_file.getnframes()
        compression = wav_file.getcomptype()
        raw_bytes = wav_file.readframes(n_frames)

    if compression != "NONE":
        raise ValueError(f"Unsupported compressed WAV: {compression}")
    if sample_width != 2:
        raise ValueError(f"Expected 16-bit PCM WAV, found sample width {sample_width}")
    if sample_rate != SAMPLE_RATE:
        raise ValueError(f"Expected {SAMPLE_RATE} Hz audio, found {sample_rate} Hz")

    audio = np.frombuffer(raw_bytes, dtype="<i2").astype(np.float32)
    if channels > 1:
        audio = audio.reshape(-1, channels).mean(axis=1)
    return audio / 32768.0


def pad_or_trim(audio: np.ndarray, target_length: int = MAX_SAMPLES) -> np.ndarray:
    if audio.shape[0] >= target_length:
        return audio[:target_length]
    padded = np.zeros(target_length, dtype=np.float32)
    padded[: audio.shape[0]] = audio
    return padded


class EmotionDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, augment: bool = False):
        self.frame = frame.reset_index(drop=True)
        self.augment = augment

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        audio = read_pcm16_mono_wav(row["processed_file_path"])
        audio = pad_or_trim(audio)

        if self.augment:
            gain = np.random.uniform(0.9, 1.1)
            noise_scale = np.random.uniform(0.0, 0.003)
            audio = audio * gain
            if noise_scale > 0:
                audio = audio + np.random.normal(0.0, noise_scale, size=audio.shape).astype(np.float32)
            audio = np.clip(audio, -1.0, 1.0)

        waveform = torch.tensor(audio, dtype=torch.float32)
        label = torch.tensor(int(row["label"]), dtype=torch.long)
        return waveform, label


train_ds = EmotionDataset(train_df, augment=True)
val_ds = EmotionDataset(val_df, augment=False)
test_ds = EmotionDataset(test_df, augment=False)

pin_memory = DEVICE.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin_memory)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory)

class_counts = train_df["label"].value_counts().sort_index().to_numpy()
class_weights = class_counts.sum() / (len(class_counts) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
class_weights

In [ ]:
class LogSpectrogram(nn.Module):
    def __init__(self, n_fft: int = 512, win_length: int = 400, hop_length: int = 160):
        super().__init__()
        self.n_fft = n_fft
        self.win_length = win_length
        self.hop_length = hop_length
        self.register_buffer("window", torch.hann_window(win_length), persistent=False)

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        spec = torch.stft(
            waveform,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.win_length,
            window=self.window,
            return_complex=True,
        )
        spec = spec.abs().clamp_min(1e-5).log()
        spec = spec.unsqueeze(1)
        return spec


class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class SERBaselineModel(nn.Module):
    def __init__(self, num_classes: int, dropout: float = DROPOUT):
        super().__init__()
        self.frontend = LogSpectrogram()
        self.encoder = nn.Sequential(
            ConvBlock(1, 16, dropout),
            ConvBlock(16, 32, dropout),
            ConvBlock(32, 64, dropout),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        x = self.frontend(waveform)
        x = self.encoder(x)
        x = self.pool(x)
        return self.head(x)


model = SERBaselineModel(num_classes=len(label_names)).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
model

In [ ]:
def run_epoch(model: nn.Module, loader: DataLoader, train: bool) -> dict:
    if train:
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    targets_all = []
    preds_all = []

    context = nullcontext() if train else torch.no_grad()
    with context:
        for waveforms, targets in loader:
            waveforms = waveforms.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            targets = targets.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))

            autocast_context = (
                torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP)
                if DEVICE.type == "cuda"
                else nullcontext()
            )

            with autocast_context:
                logits = model(waveforms)
                loss = criterion(logits, targets)

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            running_loss += loss.item() * targets.size(0)
            preds = logits.argmax(dim=1)
            targets_all.extend(targets.detach().cpu().tolist())
            preds_all.extend(preds.detach().cpu().tolist())

    avg_loss = running_loss / len(loader.dataset)
    macro_f1 = f1_score(targets_all, preds_all, average="macro")
    accuracy = float(np.mean(np.array(targets_all) == np.array(preds_all)))
    return {
        "loss": avg_loss,
        "macro_f1": macro_f1,
        "accuracy": accuracy,
        "targets": targets_all,
        "preds": preds_all,
    }


history = []
best_val_f1 = -1.0
best_checkpoint_path = ARTIFACT_DIR / "best_baseline_model.pt"

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, train=True)
    val_metrics = run_epoch(model, val_loader, train=False)
    scheduler.step(val_metrics["macro_f1"])

    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_macro_f1": train_metrics["macro_f1"],
        "train_accuracy": train_metrics["accuracy"],
        "val_loss": val_metrics["loss"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_accuracy": val_metrics["accuracy"],
        "lr": optimizer.param_groups[0]["lr"],
    })

    if val_metrics["macro_f1"] > best_val_f1:
        best_val_f1 = val_metrics["macro_f1"]
        torch.save(
            {
                "model_state": model.state_dict(),
                "label_names": label_names,
                "config": {
                    "sample_rate": SAMPLE_RATE,
                    "clip_seconds": CLIP_SECONDS,
                    "batch_size": BATCH_SIZE,
                    "epochs": EPOCHS,
                    "learning_rate": LEARNING_RATE,
                },
                "best_val_f1": best_val_f1,
            },
            best_checkpoint_path,
        )

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_metrics['loss']:.4f} train_f1={train_metrics['macro_f1']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} val_f1={val_metrics['macro_f1']:.4f}"
    )

history_df = pd.DataFrame(history)
history_df.to_csv(ARTIFACT_DIR / "training_history.csv", index=False)
history_df.tail()

In [ ]:
checkpoint = torch.load(best_checkpoint_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])

val_metrics = run_epoch(model, val_loader, train=False)
test_metrics = run_epoch(model, test_loader, train=False)

metrics_summary = {
    "best_val_f1": checkpoint["best_val_f1"],
    "val_macro_f1": val_metrics["macro_f1"],
    "val_accuracy": val_metrics["accuracy"],
    "test_macro_f1": test_metrics["macro_f1"],
    "test_accuracy": test_metrics["accuracy"],
}

with (ARTIFACT_DIR / "metrics_summary.json").open("w", encoding="utf-8") as json_file:
    json.dump(metrics_summary, json_file, indent=2)

metrics_summary

In [ ]:
report_df = pd.DataFrame(
    classification_report(
        test_metrics["targets"],
        test_metrics["preds"],
        target_names=label_names,
        output_dict=True,
        zero_division=0,
    )
).transpose()

confusion_df = pd.DataFrame(
    confusion_matrix(test_metrics["targets"], test_metrics["preds"]),
    index=[f"true_{label}" for label in label_names],
    columns=[f"pred_{label}" for label in label_names],
)

report_df.to_csv(ARTIFACT_DIR / "test_classification_report.csv")
confusion_df.to_csv(ARTIFACT_DIR / "test_confusion_matrix.csv")

report_df